<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Mock Email Server</h2>
            <span style="color:#ff7800;">This version uses a local SQLite-backed mock mailbox instead of SendGrid.<br/>
            Your agents can send, read, and reply to emails without creating real email accounts or using an external service.<br/>
            The mailbox data is persisted next to this notebook so you can inspect conversations between runs.
            </span>
        </td>
    </tr>
</table>

## Setting up the Mock Email Server

There is no external setup needed for email in this version of the notebook.

The helper file `mock_email_server.py` stores messages in a local SQLite database called `mock_email_server.sqlite3` in the same folder as this notebook.

That means your agents can:

- send emails to each other
- read inboxes by email address
- reply to existing emails with quoted history
- track which emails have been read

Everything stays local and is easy to inspect while you experiment with agent workflows.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
from mock_email_server import read_emails as read_mock_emails, send_email as send_mock_email
import asyncio

In [ ]:
load_dotenv(override=True)

In [ ]:

MOCK_SENDER_EMAIL = "sales.manager@complai.agent"
MOCK_RECIPIENT_EMAIL = "ceo@example.hu"

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to the mock recipient inbox."""
    return send_mock_email(
        MOCK_SENDER_EMAIL,
        MOCK_RECIPIENT_EMAIL,
        subject,
        html_body,
    )

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)


In [ ]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

translator_instructions = f"You are an expert to translate English text to other languages. \
You get the recipient email addess: {MOCK_RECIPIENT_EMAIL}, the subject and email body. Based on the end of the email address detect the language you translate the text to. \
For example: x.y@gmbh.de -> German, y.z@vallalat.hu -> Hungarian, etc \
If you are not sure, do not translate the texts!"

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

translator = Agent(name="Multilanguage translator from English", instructions=translator_instructions, model="gpt-4o-mini")
translator_tool = translator.as_tool(tool_name="translator", tool_description="Multilanguage translator from English")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
handoff_tools = tools = [subject_tool, html_tool, translator_tool, send_html_email]
handoff_tools

In [ ]:
instructions =instructions = f"""
You are an email translator, formatter, and sender.

You receive the plain-text body of an email to be sent to {MOCK_RECIPIENT_EMAIL}.

Follow this exact sequence:
1. Use the `subject_writer` tool to create an email subject.
2. Because the recipient domain implies a non-English language, use the `translator` tool on both the subject and the body.
3. Use the `html_converter` tool on the translated body.
4. Use `send_html_email` with the translated subject and translated HTML body.

Do not skip the `translator` step.
Do not send English text when translation is possible.
"""



emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=handoff_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")


In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

In [ ]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

# Appendix

## Test mock email service

In [ ]:
# Let's just check emails are working for you


def send_test_email():
    email = send_mock_email(
        MOCK_SENDER_EMAIL,
        MOCK_RECIPIENT_EMAIL,
        "Test email",
        "This is an important test email",
    )
    return email

# send_test_email()

In [ ]:
# Read the emails
read_mock_emails(MOCK_RECIPIENT_EMAIL, mark_as_read=False)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: add mock inbox tools so agents can read and reply to each other's emails,
            Then have the SDR keep the conversation going inside the local mailbox. This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>